# 01: Datasets and Preprocessing

This notebook explores the Rapid Evaporative Ionization Mass Spectrometry (REIMS) datasets. We will demonstrate the configuration-driven approach to data loading, perform EDA across all research datasets, and explain the preprocessing pipeline.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sys.path.append(os.path.abspath(".."))
from fishy.data.module import create_data_module

DATA_PATH = "../data/REIMS.xlsx"

## 1. Multi-Dataset Exploration
We will iterate through each dataset defined in the framework and visualize their class distributions and spectral signatures.

In [ ]:
datasets = ["species", "part", "oil", "cross-species", "batch-detection"]

for ds_name in datasets:
    dm = create_data_module(ds_name, DATA_PATH)
    dm.setup()
    df = dm.get_filtered_dataframe()
    
    print(f"--- Dataset: {ds_name} ---")
    print(f"Samples: {len(df)}, Features: {dm.get_input_dim()}")
    
    # 1. Distribution
    label_col = "Class Name"
    fig_dist = px.bar(df[label_col].value_counts(), title=f"{ds_name}: Class Distribution", labels={"index": "Class", "value": "Count"})
    fig_dist.show()
    
    # 2. Mean Spectral Signature
    X, y = dm.get_numpy_data(labels_as_indices=True)
    names = dm.get_class_names()
    feature_cols = [c for c in df.columns if c not in [label_col, "m/z"]]
    mz_axis = np.array([float(c) for c in feature_cols])
    
    fig_spec = go.Figure()
    for i, name in enumerate(names):
        mask = (y == i)
        if mask.any():
            fig_spec.add_trace(go.Scatter(x=mz_axis, y=X[mask].mean(axis=0), name=name, mode="lines"))
    
    fig_spec.update_layout(title=f"{ds_name}: Mean Spectral Signature", xaxis_title="m/z", yaxis_title="Normalized Intensity")
    fig_spec.show()

## 2. Preprocessing & Normalization

The  framework applies L2-normalization across the feature dimension (m/z peaks). This ensures that total ion count variations do not bias the machine learning models.

49373\hat{x} = rac{x}{\|x\|_2}49373

Let's visualize the effect of normalization on raw spectra.

In [ ]:
# Load raw data manually for comparison
raw_df = pd.read_excel(DATA_PATH)
raw_spectra = raw_df.iloc[:5, 1:].to_numpy() # First 5 samples

print("Raw Max Intensity:", raw_spectra.max())

# Demonstration of L2 normalization used in CustomDataset
import torch.nn.functional as F
import torch

norm_spectra = F.normalize(torch.tensor(raw_spectra, dtype=torch.float32), p=2, dim=1).numpy()
print("Normalized Max Intensity:", norm_spectra.max())

fig_comp = go.Figure()
fig_comp.add_trace(go.Scatter(y=raw_spectra[0], name="Raw Spectrum", line=dict(dash="dash")))
fig_comp.add_trace(go.Scatter(y=norm_spectra[0] * 1000, name="Normalized (scaled x1000)"))
fig_comp.update_layout(title="Comparison of Raw vs Normalized Spectrum")
fig_comp.show()